# Yelp RAG Agent — Quality Evaluation on LMDeploy Backends

**Hardware:** Colab A100 80GB  
**Prerequisite:** GitHub repo must be pushed with the new `src/yelp_rag_agent/` structure.

Runs the 20-question evaluation set against two backends and saves scored CSVs.

| Run | Config | Model | Output |
|---|---|---|---|
| fp16 | `configs/lmdeploy.yaml` | Qwen2.5-7B-Instruct | `eval_lmdeploy_fp16.csv` |
| awq  | `configs/lmdeploy.yaml` | Qwen2.5-7B-Instruct-AWQ | `eval_lmdeploy_awq.csv` |

After running, open each CSV and fill in the manual score columns,  
then use `benchmark_analysis.ipynb` to visualize results.

---
**Before running:** Set `GDRIVE_RESULTS_DIR` in Cell 2.

## Cell 1 — Install dependencies

In [ ]:
!nvidia-smi

!pip install lmdeploy -q
!pip install langchain langchain-core langgraph sentence-transformers faiss-cpu \
             transformers torch pyyaml requests pandas -q

print("\nDependencies installed.")

## Cell 2 — Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configure these paths ────────────────────────────────────────
GDRIVE_RESULTS_DIR = "/content/drive/MyDrive/yelp_rag_results"
GDRIVE_VECTORSTORE = "/content/drive/MyDrive/yelp_rag_vectorstore"
GDRIVE_DATA        = "/content/drive/MyDrive/yelp_rag_data"
REPO_URL           = "https://github.com/your-username/yelp-rag-agent-deployment"  # ← update
# ─────────────────────────────────────────────────────────────────

import os, subprocess, sys, shutil, time

REPO_DIR = "/content/yelp-rag-agent-deployment"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# Verify new package structure exists
pkg_path = os.path.join(REPO_DIR, "src", "yelp_rag_agent")
if os.path.exists(pkg_path):
    print(f"Package found at {pkg_path}")
else:
    raise RuntimeError(
        f"src/yelp_rag_agent/ not found at {pkg_path}.\n"
        "Make sure the repo has been pushed with the new structure."
    )

# Add src/ to path and install package
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
!pip install -e {REPO_DIR} --no-deps -q

# Link large files from Drive
def symlink_if_exists(src, dst):
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)
        print(f"  Linked: {dst}")
    elif not os.path.exists(src):
        print(f"  WARNING: not found: {src}")

os.makedirs(f"{REPO_DIR}/vectorstore", exist_ok=True)
os.makedirs(f"{REPO_DIR}/data/processed", exist_ok=True)
os.makedirs(f"{REPO_DIR}/data/raw", exist_ok=True)
os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
os.makedirs(GDRIVE_RESULTS_DIR, exist_ok=True)

symlink_if_exists(f"{GDRIVE_VECTORSTORE}/review_chunks.index",
                  f"{REPO_DIR}/vectorstore/review_chunks.index")
symlink_if_exists(f"{GDRIVE_VECTORSTORE}/review_chunks.pkl",
                  f"{REPO_DIR}/vectorstore/review_chunks.pkl")
symlink_if_exists(f"{GDRIVE_DATA}/yelp_reviews_sampled_50k.csv",
                  f"{REPO_DIR}/data/processed/yelp_reviews_sampled_50k.csv")
symlink_if_exists(f"{GDRIVE_DATA}/yelp_academic_dataset_business.json",
                  f"{REPO_DIR}/data/raw/yelp_academic_dataset_business.json")

print("\nSetup complete.")

## Cell 3 — Verify package import

In [ ]:
from yelp_rag_agent.config import PROJECT_ROOT, VECTORSTORE_INDEX, DATA_PATH
from yelp_rag_agent.backends import load_backend
from yelp_rag_agent.tools.summarizer_tool import set_backend

print(f"PROJECT_ROOT     : {PROJECT_ROOT}")
print(f"VECTORSTORE_INDEX: {VECTORSTORE_INDEX} — exists={VECTORSTORE_INDEX.exists()}")
print(f"DATA_PATH        : {DATA_PATH} — exists={DATA_PATH.exists()}")
print("\nPackage import OK.")

## Cell 4 — Server helpers

In [ ]:
import requests

BASE_URL = "http://localhost:23333"

def wait_for_server(base_url: str, timeout: int = 120, poll_interval: int = 5) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if requests.get(f"{base_url}/v1/models", timeout=5).status_code == 200:
                return True
        except Exception:
            pass
        time.sleep(poll_interval)
    return False

def start_server(mode: str):
    cmd  = ["bash", f"{REPO_DIR}/scripts/serve_lmdeploy.sh", mode]
    proc = subprocess.Popen(
        cmd,
        stdout=open(f"/tmp/lmdeploy_{mode}.log", "w"),
        stderr=subprocess.STDOUT,
    )
    print(f"Server starting (mode={mode}, pid={proc.pid}) ...")
    if wait_for_server(BASE_URL, timeout=120):
        print("Server is ready.")
    else:
        print("WARNING: server not ready. Log tail:")
        with open(f"/tmp/lmdeploy_{mode}.log") as f:
            print(f.read()[-2000:])
    return proc

def stop_server(proc):
    proc.terminate()
    proc.wait()
    time.sleep(15)
    print("Server stopped.")

print("Server helpers ready.")

## Cell 5 — Run quality eval: fp16

In [ ]:
from yelp_rag_agent.backends import load_backend
from yelp_rag_agent.tools.summarizer_tool import set_backend
from yelp_rag_agent.evaluation.run_eval import run_evaluation

OUTPUT_FP16 = "eval_lmdeploy_fp16.csv"

proc_fp16 = start_server("fp16")
try:
    backend = load_backend(
        f"{REPO_DIR}/configs/lmdeploy.yaml",
        overrides={"model": "Qwen/Qwen2.5-7B-Instruct"},
    )
    set_backend(backend)
    run_evaluation(
        config_path = f"{REPO_DIR}/configs/lmdeploy.yaml",
        output_name = OUTPUT_FP16,
    )
finally:
    stop_server(proc_fp16)

# Backup to Drive
local_csv = f"{REPO_DIR}/results/{OUTPUT_FP16}"
if os.path.exists(local_csv):
    shutil.copy(local_csv, f"{GDRIVE_RESULTS_DIR}/{OUTPUT_FP16}")
    print(f"Backed up: {GDRIVE_RESULTS_DIR}/{OUTPUT_FP16}")

print("\nfp16 quality eval complete.")

## Cell 6 — Run quality eval: AWQ

In [ ]:
from yelp_rag_agent.backends import load_backend
from yelp_rag_agent.tools.summarizer_tool import set_backend
from yelp_rag_agent.evaluation.run_eval import run_evaluation

OUTPUT_AWQ = "eval_lmdeploy_awq.csv"

proc_awq = start_server("awq")
try:
    backend = load_backend(
        f"{REPO_DIR}/configs/lmdeploy.yaml",
        overrides={"model": "Qwen/Qwen2.5-7B-Instruct-AWQ"},
    )
    set_backend(backend)
    run_evaluation(
        config_path = f"{REPO_DIR}/configs/lmdeploy.yaml",
        output_name = OUTPUT_AWQ,
    )
finally:
    stop_server(proc_awq)

# Backup to Drive
local_csv = f"{REPO_DIR}/results/{OUTPUT_AWQ}"
if os.path.exists(local_csv):
    shutil.copy(local_csv, f"{GDRIVE_RESULTS_DIR}/{OUTPUT_AWQ}")
    print(f"Backed up: {GDRIVE_RESULTS_DIR}/{OUTPUT_AWQ}")

print("\nAWQ quality eval complete.")

## Cell 7 — Preview results

In [ ]:
import pandas as pd

for fname in [OUTPUT_FP16, OUTPUT_AWQ]:
    path = f"{REPO_DIR}/results/{fname}"
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"\n{fname}: {len(df)} rows")
        print(df[["system", "question_type", "tool_count", "elapsed_seconds",
                   "has_evidence", "answer_length"]].to_string(index=False))
    else:
        print(f"{fname}: not found")

print("\nNext step: open the CSVs, fill in score columns,")
print("then run benchmark_analysis.ipynb to visualize results.")